# Notebook 14 — Additional Analysis
**SeasonalNaive baselines · CV-RMSE · VIF · Diebold-Mariano tests · Verification checks**

| Task | Content |
|------|---------|
| 1 | SeasonalNaive-364 at Olympia and Erhardt |
| 2 | CV-RMSE and skill scores — all models × stations |
| 3 | VIF for SARIMAX weather regressors + univariate vs multivariate temperature coefficient |
| 4 | Diebold-Mariano tests — 4 pairs × 3 stations |
| 5a | Verification: sunshine-only SARIMAX at Hirsch |
| 5b | Hybrid XGBoost corrector feature importances |
| 5c | Erhardt monthly demand — training vs test period |

In [ ]:
# ── figure setup (nb14 figures) ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10,
})
sns.set_theme(style='whitegrid')

FIGURES = Path('../results/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
NB = 'nb14'
MONTHS_SHORT = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
SPLIT_COLORS = {'Train': '#3b82f6', 'Val': '#f59e0b', 'Test': '#ef4444'}
GAP_START = pd.Timestamp('2024-09-06')
GAP_END   = pd.Timestamp('2024-11-26')

def savefig(name):
    full = f'{NB}_{name}'
    plt.savefig(FIGURES / f'{full}.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  saved → results/figures/{full}.png')

print('nb14 figure setup complete.')


In [1]:
import pickle, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from xgboost import XGBRegressor
import holidays

logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

TABLES  = Path('../results/tables')
MODELS  = Path('../results/models')
TABLES.mkdir(parents=True, exist_ok=True)

STATIONS      = ['Hirsch', 'Olympia', 'Erhardt']
WEATHER_COLS  = ['max_temp', 'sunshine_hours', 'precipitation']
TRAIN_START   = '2013-01-01'
TRAIN_END     = '2022-12-31'
VAL_START     = '2023-01-01'
VAL_END       = '2023-12-31'
TEST_START    = pd.Timestamp('2024-01-01')
TEST_END      = pd.Timestamp('2026-04-30')
ANNUAL_PERIOD = 365.25
REF_DATE      = pd.Timestamp('2013-01-01')
K             = 4
BY_HOLIDAYS   = set(holidays.Germany(state='BY', years=range(2013, 2027)).keys())
_by_hols = holidays.Germany(state='BY', years=range(2013, 2027))
holiday_df = pd.DataFrame([
    {'ds': pd.Timestamp(d), 'holiday': name, 'lower_window': 0, 'upper_window': 0}
    for d, name in _by_hols.items()
])

NB = 'nb14'
print('Setup complete.')

/Users/sousadaniela/time-series-rad/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


Setup complete.


In [2]:
def rolling_rmse_mae(y_true, y_pred, horizon=7):
    """Non-overlapping rolling window RMSE/MAE — matches all other notebooks."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rmses, maes = [], []
    for start in range(0, len(y_true) - horizon + 1, horizon):
        sl = slice(start, start + horizon)
        err = y_true[sl] - y_pred[sl]
        rmses.append(np.sqrt(np.mean(err**2)))
        maes.append(np.mean(np.abs(err)))
    return float(np.mean(rmses)), float(np.mean(maes))


def dm_test(e1, e2, h=7):
    """Harvey-Leybourne-Newbold (1997) modified DM test, squared loss.
    d_t = e1_t^2 - e2_t^2. H0: equal predictive accuracy.
    Positive DM => model 2 significantly better (lower loss).
    Negative DM => model 1 significantly better.
    """
    e1, e2 = np.asarray(e1, dtype=float), np.asarray(e2, dtype=float)
    d = e1**2 - e2**2
    n = len(d)
    d_bar = d.mean()
    # Newey-West variance with lag truncation h-1
    gamma_0 = np.var(d, ddof=1)
    gamma_sum = sum(
        (1 - k / h) * np.mean((d[k:] - d_bar) * (d[:-k] - d_bar))
        for k in range(1, h)
    )
    var_d = (gamma_0 + 2 * gamma_sum) / n
    if var_d <= 0:
        return np.nan, np.nan
    DM = d_bar / np.sqrt(var_d)
    # HLN finite-sample correction
    k_hlm = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    DM_star = DM * k_hlm
    p_val = 2 * stats.t.sf(abs(DM_star), df=n - 1)
    return float(DM_star), float(p_val)


def load_station(station):
    """Load and prep a station's data (log1p precip, interpolate Olympia)."""
    df = pd.read_csv('../data/processed/master_bike_data.csv', parse_dates=['date'])
    raw = (df[df['station'] == station]
           .set_index('date').sort_index()[['total'] + WEATHER_COLS])
    raw.index = pd.DatetimeIndex(raw.index)
    if station == 'Olympia':
        raw = raw.interpolate(method='time', limit=7)
    raw = raw.dropna(subset=WEATHER_COLS)
    return raw


def build_fourier(index):
    t = (index - REF_DATE).days
    cols = {}
    for k in range(1, K+1):
        cols[f'sin_{k}'] = np.sin(2*np.pi*k*t/ANNUAL_PERIOD)
        cols[f'cos_{k}'] = np.cos(2*np.pi*k*t/ANNUAL_PERIOD)
    return pd.DataFrame(cols, index=index)


def build_sarimax_exog(index, df_weather):
    four = build_fourier(index)
    wx = df_weather[WEATHER_COLS].reindex(index)
    wx = wx.copy()
    wx['precipitation'] = np.log1p(wx['precipitation'])
    return pd.concat([four, wx], axis=1)


def build_correction_features(index, df_weather):
    feat = pd.DataFrame(index=index)
    feat['dow']        = index.dayofweek
    feat['month']      = index.month
    feat['doy']        = index.dayofyear
    feat['is_weekend'] = (index.dayofweek >= 5).astype(int)
    feat['is_holiday'] = index.normalize().isin(
        pd.DatetimeIndex([pd.Timestamp(d) for d in BY_HOLIDAYS])).astype(int)
    wx = df_weather[WEATHER_COLS].reindex(index).copy()
    wx['precipitation'] = np.log1p(wx['precipitation'])
    feat['max_temp']       = wx['max_temp'].values
    feat['sunshine_hours'] = wx['sunshine_hours'].values
    feat['precipitation']  = wx['precipitation'].values
    four = build_fourier(index)
    return pd.concat([feat, four], axis=1)


print('Helpers defined.')

Helpers defined.


## Task 1 — SeasonalNaive-364: Olympia and Erhardt
Prediction = value observed 364 days prior. Applied to val and test periods.

In [3]:
sn364_results = {}

for station in ['Olympia', 'Erhardt']:
    raw = load_station(station)

    val_idx  = raw[(raw.index >= VAL_START)  & (raw.index <= VAL_END)].index
    test_idx = raw[(raw.index >= TEST_START) & (raw.index <= TEST_END)].index

    def sn364_preds(idx):
        preds = []
        for d in idx:
            lookback = d - pd.Timedelta(days=364)
            if lookback in raw.index:
                preds.append(raw.loc[lookback, 'total'])
            else:
                preds.append(np.nan)
        return np.array(preds)

    val_preds  = sn364_preds(val_idx)
    test_preds = sn364_preds(test_idx)

    val_actual  = raw.loc[val_idx,  'total'].values
    test_actual = raw.loc[test_idx, 'total'].values

    # Drop NaN pairs
    vok = ~np.isnan(val_preds)
    tok = ~np.isnan(test_preds)

    val_rmse,  val_mae  = rolling_rmse_mae(val_actual[vok],  val_preds[vok])
    test_rmse, test_mae = rolling_rmse_mae(test_actual[tok], test_preds[tok])

    sn364_results[station] = {
        'val_rmse': round(val_rmse, 1), 'val_mae': round(val_mae, 1),
        'test_rmse': round(test_rmse, 1), 'test_mae': round(test_mae, 1),
    }
    print(f'{station} SeasonalNaive-364:')
    print(f'  Val  7d RMSE: {val_rmse:.1f}  MAE: {val_mae:.1f}')
    print(f'  Test 7d RMSE: {test_rmse:.1f}  MAE: {test_mae:.1f}')

# Add Hirsch from nb03
sn364_results['Hirsch'] = {'val_rmse': 614.0, 'test_rmse': 662.1}

Olympia SeasonalNaive-364:
  Val  7d RMSE: 865.9  MAE: 727.6
  Test 7d RMSE: 972.5  MAE: 804.1
Erhardt SeasonalNaive-364:
  Val  7d RMSE: 1939.7  MAE: 1620.8
  Test 7d RMSE: 1906.5  MAE: 1601.3


## Task 2 — CV-RMSE and Skill Scores
CV-RMSE = (Test 7d RMSE / mean test-period demand) × 100
Skill = (1 − RMSE_model / RMSE_SN364) × 100

In [4]:
# Load test-period mean demand
df_raw = pd.read_csv('../data/processed/master_bike_data.csv', parse_dates=['date'])
test_means = {}
for s in STATIONS:
    test_means[s] = df_raw[(df_raw['station'] == s) &
                           (df_raw['date'] >= TEST_START) &
                           (df_raw['date'] <= TEST_END)]['total'].mean()
print('Test-period mean demand:')
for s, v in test_means.items():
    print(f'  {s}: {v:.1f}')

# Load all RMSE values from upstream CSVs
ult = pd.read_csv(TABLES / 'ultimate_comparison.csv').set_index('Station')

rmse_table = {
    'SARIMAX+wx':   {s: ult.loc[s, 'SARIMAX_wx']  for s in STATIONS},
    'Prophet+wx':   {s: ult.loc[s, 'Prophet_wx']  for s in STATIONS},
    'SARIMAX+XGB':  {s: ult.loc[s, 'SARIMAX_XGB'] for s in STATIONS},
    'Prophet+XGB':  {s: ult.loc[s, 'Prophet_XGB'] for s in STATIONS},
    'SeasonalNaive-364': {s: sn364_results[s]['test_rmse'] for s in STATIONS},
}

sn_rmse = rmse_table['SeasonalNaive-364']

print('\n=== CV-RMSE (%) ===')
rows_cv = []
for model, rmses in rmse_table.items():
    row = {'Model': model}
    for s in STATIONS:
        row[s] = round(rmses[s] / test_means[s] * 100, 1)
    rows_cv.append(row)
cv_df = pd.DataFrame(rows_cv).set_index('Model')
print(cv_df.to_string())

print('\n=== Skill Score (%) vs SeasonalNaive-364 ===')
rows_sk = []
for model, rmses in rmse_table.items():
    row = {'Model': model}
    for s in STATIONS:
        row[s] = round((1 - rmses[s] / sn_rmse[s]) * 100, 1)
    rows_sk.append(row)
sk_df = pd.DataFrame(rows_sk).set_index('Model')
print(sk_df.to_string())

# Save
cv_df.to_csv(TABLES / 'nb14_cvrmse.csv')
sk_df.to_csv(TABLES / 'nb14_skill_scores.csv')
print('\nSaved: nb14_cvrmse.csv, nb14_skill_scores.csv')

Test-period mean demand:
  Hirsch: 1490.0
  Olympia: 2201.4
  Erhardt: 3949.6

=== CV-RMSE (%) ===
                   Hirsch  Olympia  Erhardt
Model                                      
SARIMAX+wx           31.6     34.5     21.4
Prophet+wx           21.2     24.6     22.0
SARIMAX+XGB          28.1     33.0     18.2
Prophet+XGB          18.8     22.7     19.6
SeasonalNaive-364    44.4     44.2     48.3

=== Skill Score (%) vs SeasonalNaive-364 ===
                   Hirsch  Olympia  Erhardt
Model                                      
SARIMAX+wx           28.9     21.9     55.8
Prophet+wx           52.2     44.2     54.3
SARIMAX+XGB          36.7     25.2     62.2
Prophet+XGB          57.7     48.7     59.4
SeasonalNaive-364     0.0      0.0      0.0

Saved: nb14_cvrmse.csv, nb14_skill_scores.csv


## Task 3 — VIF for SARIMAX Weather Regressors
Design matrix: Fourier K=4 (8 terms) + max_temp + sunshine_hours + log1p(precipitation), trainval period.

In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_results = {}

for station in ['Hirsch', 'Olympia']:
    raw = load_station(station)
    tv  = raw[(raw.index >= TRAIN_START) & (raw.index <= VAL_END)].copy()
    exog = build_sarimax_exog(tv.index, tv)
    exog = exog.dropna()
    X = exog.values
    feature_names = exog.columns.tolist()

    vifs = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    vif_df = pd.DataFrame({'Feature': feature_names, 'VIF': [round(v, 2) for v in vifs]})
    vif_results[station] = vif_df

    print(f'\n--- VIF: {station} (trainval, Fourier K=4 + weather) ---')
    print(vif_df.to_string(index=False))

# Save combined
for s, df in vif_results.items():
    df.to_csv(TABLES / f'nb14_vif_{s.lower()}.csv', index=False)


--- VIF: Hirsch (trainval, Fourier K=4 + weather) ---
       Feature  VIF
         sin_1 1.12
         cos_1 1.29
         sin_2 1.00
         cos_2 1.01
         sin_3 1.00
         cos_3 1.00
         sin_4 1.00
         cos_4 1.00
      max_temp 7.94
sunshine_hours 5.69
 precipitation 1.79



--- VIF: Olympia (trainval, Fourier K=4 + weather) ---
       Feature  VIF
         sin_1 1.12
         cos_1 1.29
         sin_2 1.00
         cos_2 1.01
         sin_3 1.00
         cos_3 1.00
         sin_4 1.00
         cos_4 1.00
      max_temp 7.94
sunshine_hours 5.69
 precipitation 1.79


In [6]:
# Univariate vs multivariate max_temp coefficient (train-only, matches nb05/06)
print('=== Univariate vs Multivariate max_temp Coefficient (train-only fit) ===')
print()
coef_rows = []
for station in ['Hirsch', 'Olympia']:
    raw = load_station(station)
    tr  = raw[(raw.index >= TRAIN_START) & (raw.index <= TRAIN_END)].copy()
    four = build_fourier(tr.index)
    tr_log = tr.copy()
    tr_log['precipitation'] = np.log1p(tr_log['precipitation'])

    for label, wx_cols in [('univariate (temp only)', ['max_temp']),
                            ('full (all weather)',    WEATHER_COLS)]:
        exog = pd.concat([four, tr_log[wx_cols]], axis=1)
        m = SARIMAX(tr['total'], exog=exog, order=(2,1,2), seasonal_order=(1,0,2,7),
                    enforce_stationarity=False, enforce_invertibility=False)
        res = m.fit(disp=False)
        coef = round(res.params['max_temp'], 1)
        se   = round(res.bse['max_temp'], 2)
        print(f'  {station} {label}: coef={coef:+.1f}  SE={se}')
        coef_rows.append({'Station': station, 'Spec': label, 'max_temp_coef': coef, 'SE': se})
    print()

coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(TABLES / 'nb14_temp_coef_comparison.csv', index=False)

=== Univariate vs Multivariate max_temp Coefficient (train-only fit) ===



  Hirsch univariate (temp only): coef=+44.3  SE=1.58


  Hirsch full (all weather): coef=+26.7  SE=1.84



  Olympia univariate (temp only): coef=+88.2  SE=2.87


  Olympia full (all weather): coef=+41.6  SE=3.0



## Task 4 — Diebold-Mariano Tests
Pairs tested at each station:
- (a) SARIMAX+wx vs Prophet+wx
- (b) SARIMAX+XGB vs Prophet+XGB
- (c) Prophet+wx vs Prophet+XGB (hybrid gain)
- (d) SARIMAX+wx vs SARIMAX+XGB (hybrid gain)

DM statistic sign: positive => model 2 is better. h=7, Newey-West lag truncation=6.

In [7]:
# Generate forecast error series for all 4 models × 3 stations
# errors[station][model] = array of daily errors (actual - predicted)

errors = {s: {} for s in STATIONS}

for station in STATIONS:
    print(f'=== {station} ===')
    raw = load_station(station)
    tv_raw  = raw[(raw.index >= TRAIN_START) & (raw.index <= VAL_END)]
    test_raw = raw[(raw.index >= TEST_START)  & (raw.index <= TEST_END)]
    test_idx  = test_raw.index
    actual_t  = test_raw['total'].values

    # ── SARIMAX+wx ──────────────────────────────────────────────────────
    with open(MODELS / f'{station.lower()}_sarimax_final.pkl', 'rb') as f:
        sar_res = pickle.load(f)

    if station == 'Erhardt':
        full_idx = pd.date_range(TEST_START, TEST_END, freq='D')
        exog_full = build_sarimax_exog(full_idx, raw.reindex(full_idx, method='nearest'))
        fc_full = sar_res.forecast(steps=len(full_idx), exog=exog_full)
        sar_preds = pd.Series(fc_full.values, index=full_idx).reindex(test_idx).values
    else:
        exog_test = build_sarimax_exog(test_idx, raw)
        fc = sar_res.forecast(steps=len(test_idx), exog=exog_test)
        sar_preds = fc.values

    errors[station]['sarimax_wx'] = actual_t - sar_preds
    sar_rmse, _ = rolling_rmse_mae(actual_t, sar_preds)
    print(f'  SARIMAX+wx  RMSE={sar_rmse:.1f}')

    # ── Prophet+wx ──────────────────────────────────────────────────────
    dp = tv_raw.reset_index().rename(columns={'date': 'ds', 'total': 'y'})
    dp['precipitation'] = np.log1p(dp['precipitation'])
    dp_test = test_raw.reset_index().rename(columns={'date': 'ds', 'total': 'y'})
    dp_test['precipitation'] = np.log1p(dp_test['precipitation'])

    m_proph = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True,
                      weekly_seasonality=True, daily_seasonality=False,
                      changepoint_prior_scale=0.05, holidays=holiday_df)
    for col in WEATHER_COLS:
        m_proph.add_regressor(col)
    m_proph.fit(dp)
    proph_fc = m_proph.predict(dp_test[['ds'] + WEATHER_COLS])
    proph_preds = proph_fc['yhat'].values

    errors[station]['prophet_wx'] = actual_t - proph_preds
    proph_rmse, _ = rolling_rmse_mae(actual_t, proph_preds)
    print(f'  Prophet+wx  RMSE={proph_rmse:.1f}')

    # ── SARIMAX+XGB hybrid ───────────────────────────────────────────────
    fitted = sar_res.fittedvalues
    fitted.index = pd.DatetimeIndex(fitted.index)
    tv_actual = raw.loc[raw.index.isin(fitted.index), 'total'].reindex(fitted.index).values
    resid_tv  = tv_actual - fitted.values
    Xc_tv = build_correction_features(fitted.index, raw.reindex(fitted.index))
    valid_tv = ~Xc_tv.isna().any(axis=1) & ~np.isnan(resid_tv)
    corr_sar = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
    corr_sar.fit(Xc_tv[valid_tv], resid_tv[valid_tv])
    Xc_test = build_correction_features(test_idx, raw.reindex(test_idx))
    valid_t  = ~Xc_test.isna().any(axis=1)
    correction = np.zeros(len(test_idx))
    correction[valid_t.values] = corr_sar.predict(Xc_test[valid_t])
    sar_hyb_preds = sar_preds + correction

    errors[station]['sarimax_xgb'] = actual_t - sar_hyb_preds
    sarxgb_rmse, _ = rolling_rmse_mae(actual_t, sar_hyb_preds)
    print(f'  SARIMAX+XGB RMSE={sarxgb_rmse:.1f}')

    # ── Prophet+XGB hybrid ──────────────────────────────────────────────
    tv_idx_p   = pd.DatetimeIndex(dp['ds'])
    tv_fitted_p = m_proph.predict(dp[['ds'] + WEATHER_COLS])['yhat'].values
    resid_p_tv  = dp['y'].values - tv_fitted_p
    Xc_tv_p     = build_correction_features(tv_idx_p, raw.reindex(tv_idx_p))
    valid_tv_p  = ~Xc_tv_p.isna().any(axis=1) & ~np.isnan(resid_p_tv)
    corr_proph  = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
    corr_proph.fit(Xc_tv_p[valid_tv_p], resid_p_tv[valid_tv_p])
    valid_tp = ~Xc_test.isna().any(axis=1)
    correction_p = np.zeros(len(test_idx))
    correction_p[valid_tp.values] = corr_proph.predict(Xc_test[valid_tp])
    proph_hyb_preds = proph_preds + correction_p

    errors[station]['prophet_xgb'] = actual_t - proph_hyb_preds
    prophxgb_rmse, _ = rolling_rmse_mae(actual_t, proph_hyb_preds)
    print(f'  Prophet+XGB RMSE={prophxgb_rmse:.1f}')
    print()

    # Store correctors for task 5b
    errors[station]['_corr_sar']   = corr_sar
    errors[station]['_corr_proph'] = corr_proph
    errors[station]['_feat_names'] = list(Xc_test.columns)

print('All forecast errors generated.')

=== Hirsch ===


  SARIMAX+wx  RMSE=467.5


16:18:34 - cmdstanpy - INFO - Chain [1] start processing


16:18:35 - cmdstanpy - INFO - Chain [1] done processing


  Prophet+wx  RMSE=316.5


...

16:18:39 - cmdstanpy - INFO - Chain [1] start processing


16:18:39 - cmdstanpy - INFO - Chain [1] done processing


  Prophet+wx  RMSE=870.5


  SARIMAX+XGB RMSE=719.9


  Prophet+XGB RMSE=774.4

All forecast errors generated.


In [8]:
# Diebold-Mariano tests
# Pairs: (e1_key, e2_key, label_e1, label_e2)
# DM > 0 => model 2 significantly better
pairs = [
    ('sarimax_wx',  'prophet_wx',  'SARIMAX+wx',  'Prophet+wx',  '(a) SARIMAX+wx vs Prophet+wx'),
    ('sarimax_xgb', 'prophet_xgb', 'SARIMAX+XGB', 'Prophet+XGB', '(b) SARIMAX+XGB vs Prophet+XGB'),
    ('prophet_wx',  'prophet_xgb', 'Prophet+wx',  'Prophet+XGB', '(c) Prophet+wx vs Prophet+XGB (hybrid gain)'),
    ('sarimax_wx',  'sarimax_xgb', 'SARIMAX+wx',  'SARIMAX+XGB', '(d) SARIMAX+wx vs SARIMAX+XGB (hybrid gain)'),
]

col_pair, col_sta, col_dm, col_pv = 'Pair', 'Station', 'DM*', 'p-val'
header = f'{col_pair:<50} {col_sta:<10} {col_dm:>8} {col_pv:>8} Interpretation'
dm_rows = []
print(header)
print('-' * 110)

for e1k, e2k, lbl1, lbl2, pair_label in pairs:
    for station in STATIONS:
        e1 = errors[station][e1k]
        e2 = errors[station][e2k]
        dm, pval = dm_test(e1, e2, h=7)
        if np.isnan(dm):
            interp = 'n/a'
        elif pval > 0.05:
            interp = f'No sig. diff (p={pval:.3f})'
        elif dm > 0:
            interp = f'{lbl2} significantly better (p={pval:.3f})'
        else:
            interp = f'{lbl1} significantly better (p={pval:.3f})'
        row_str = f'{pair_label:<50} {station:<10} {dm:>+8.3f} {pval:>8.4f} {interp}'
        print(row_str)
        dm_rows.append({
            'Pair': pair_label, 'Station': station,
            'e1': lbl1, 'e2': lbl2,
            'DM_stat': round(dm, 3), 'p_value': round(pval, 4),
            'Interpretation': interp
        })
    print()

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv(TABLES / 'nb14_dm_tests.csv', index=False)
print('Saved: nb14_dm_tests.csv')


Pair                                               Station         DM*    p-val Interpretation
--------------------------------------------------------------------------------------------------------------
(a) SARIMAX+wx vs Prophet+wx                       Hirsch      +10.113   0.0000 Prophet+wx significantly better (p=0.000)
(a) SARIMAX+wx vs Prophet+wx                       Olympia      +6.394   0.0000 Prophet+wx significantly better (p=0.000)
(a) SARIMAX+wx vs Prophet+wx                       Erhardt      -0.729   0.4663 No sig. diff (p=0.466)

(b) SARIMAX+XGB vs Prophet+XGB                     Hirsch      +10.367   0.0000 Prophet+XGB significantly better (p=0.000)
(b) SARIMAX+XGB vs Prophet+XGB                     Olympia      +6.508   0.0000 Prophet+XGB significantly better (p=0.000)
(b) SARIMAX+XGB vs Prophet+XGB                     Erhardt      -2.406   0.0164 SARIMAX+XGB significantly better (p=0.016)

(c) Prophet+wx vs Prophet+XGB (hybrid gain)        Hirsch       +5.539   0.0

## Task 5 — Verification Checks

In [9]:
# Task 5a: Sunshine-only SARIMAX at Hirsch
print('=== Task 5a: Sunshine-only SARIMAX at Hirsch ===')
raw_h = load_station('Hirsch')
tv_h  = raw_h[(raw_h.index >= TRAIN_START) & (raw_h.index <= VAL_END)]
test_h = raw_h[(raw_h.index >= TEST_START) & (raw_h.index <= TEST_END)]

four_tv   = build_fourier(tv_h.index)
four_test = build_fourier(test_h.index)

tv_sun_log = tv_h.copy()
tv_sun_log['precipitation'] = np.log1p(tv_sun_log['precipitation'])
test_sun_log = test_h.copy()
test_sun_log['precipitation'] = np.log1p(test_sun_log['precipitation'])

exog_tv_sun   = pd.concat([four_tv,   tv_sun_log[['sunshine_hours']]], axis=1)
exog_test_sun = pd.concat([four_test, test_sun_log[['sunshine_hours']]], axis=1)

m_sun = SARIMAX(tv_h['total'], exog=exog_tv_sun, order=(2,1,2), seasonal_order=(1,0,2,7),
                enforce_stationarity=False, enforce_invertibility=False)
res_sun = m_sun.fit(disp=False)

fc_sun = res_sun.forecast(steps=len(test_h), exog=exog_test_sun)
sun_rmse, sun_mae = rolling_rmse_mae(test_h['total'].values, fc_sun.values)

print(f'  sunshine-only SARIMAX+Fourier K=4: Test 7d RMSE = {sun_rmse:.1f}  MAE = {sun_mae:.1f}')
print(f'  sunshine coef = {res_sun.params["sunshine_hours"]:.1f}  SE={res_sun.bse["sunshine_hours"]:.2f}')
print(f'  (Reference: full-model sunshine coef = 37.5)')
print(f'  (Report expected ~479; actual = {sun_rmse:.1f})')

=== Task 5a: Sunshine-only SARIMAX at Hirsch ===


  sunshine-only SARIMAX+Fourier K=4: Test 7d RMSE = 446.7  MAE = 379.8
  sunshine coef = 57.2  SE=1.43
  (Reference: full-model sunshine coef = 37.5)
  (Report expected ~479; actual = 446.7)


In [10]:
# Task 5b: Hybrid corrector feature importances
print('=== Task 5b: Hybrid XGBoost Corrector Feature Importances ===')
print('(Top 5 by gain importance)')
print()

# Load standalone XGBoost importances from nb11
xgb_perf = pd.read_csv(TABLES / 'xgboost_performance.csv')

importance_rows = []
for station in STATIONS:
    feat_names = errors[station]['_feat_names']

    for hybrid_label, corr_key in [('SARIMAX+XGB corrector', '_corr_sar'),
                                    ('Prophet+XGB corrector', '_corr_proph')]:
        corr = errors[station][corr_key]
        imp = corr.feature_importances_
        top5_idx = np.argsort(imp)[::-1][:5]
        top5 = [(feat_names[i], round(imp[i], 4)) for i in top5_idx]
        print(f'  {station} — {hybrid_label}:')
        for rank, (feat, val) in enumerate(top5, 1):
            print(f'    {rank}. {feat}: {val:.4f}')
            importance_rows.append({
                'Station': station, 'Model': hybrid_label,
                'Rank': rank, 'Feature': feat, 'Importance': val
            })
        print()

imp_df = pd.DataFrame(importance_rows)
imp_df.to_csv(TABLES / 'nb14_hybrid_importances.csv', index=False)

=== Task 5b: Hybrid XGBoost Corrector Feature Importances ===
(Top 5 by gain importance)

  Hirsch — SARIMAX+XGB corrector:
    1. is_holiday: 0.3799
    2. is_weekend: 0.0840
    3. cos_1: 0.0558
    4. dow: 0.0522
    5. max_temp: 0.0467

  Hirsch — Prophet+XGB corrector:
    1. is_weekend: 0.2135
    2. dow: 0.1015
    3. doy: 0.0688
    4. is_holiday: 0.0669
    5. sin_1: 0.0635

  Olympia — SARIMAX+XGB corrector:
    1. is_holiday: 0.1036
    2. doy: 0.0775
    3. month: 0.0751
    4. sin_1: 0.0722
    5. is_weekend: 0.0720

  Olympia — Prophet+XGB corrector:
    1. is_weekend: 0.1016
    2. cos_1: 0.0789
    3. sin_1: 0.0778
    4. dow: 0.0735
    5. sunshine_hours: 0.0686

  Erhardt — SARIMAX+XGB corrector:
    1. is_holiday: 0.2347
    2. sunshine_hours: 0.0821
    3. cos_1: 0.0691
    4. doy: 0.0661
    5. max_temp: 0.0659

  Erhardt — Prophet+XGB corrector:
    1. sunshine_hours: 0.0927
    2. max_temp: 0.0914
    3. cos_2: 0.0880
    4. is_weekend: 0.0794
    5. cos_1: 0.076

In [11]:
# Task 5c: Erhardt monthly averages — training vs test
print('=== Task 5c: Erhardt Monthly Mean Demand ===')
df_raw2 = pd.read_csv('../data/processed/master_bike_data.csv', parse_dates=['date'])
erhardt = df_raw2[df_raw2['station'] == 'Erhardt'].set_index('date').sort_index()

train_e = erhardt[(erhardt.index >= TRAIN_START) & (erhardt.index <= TRAIN_END)]
test_e  = erhardt[(erhardt.index >= TEST_START)  & (erhardt.index <= TEST_END)]

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

train_monthly = train_e.groupby(train_e.index.month)['total'].mean().round(0)
test_monthly  = test_e.groupby(test_e.index.month)['total'].mean().round(0)

monthly_df = pd.DataFrame({
    'Month': month_names,
    'Train mean (2013-22)': [train_monthly.get(m, np.nan) for m in range(1,13)],
    'Test mean (2024-26)':  [test_monthly.get(m,  np.nan) for m in range(1,13)],
})
monthly_df['Change %'] = ((monthly_df['Test mean (2024-26)'] - monthly_df['Train mean (2013-22)'])
                          / monthly_df['Train mean (2013-22)'] * 100).round(1)

print(monthly_df.to_string(index=False))
print()
print('Flagged months:')
for month_n, m_idx in [('April', 4), ('June', 6), ('July', 7), ('August', 8)]:
    row = monthly_df.iloc[m_idx - 1]
    print(f'  {month_n}: train={row["Train mean (2013-22)"]:.0f}  test={row["Test mean (2024-26)"]:.0f}  change={row["Change %"]:+.1f}%')

monthly_df.to_csv(TABLES / 'nb14_erhardt_monthly.csv', index=False)

=== Task 5c: Erhardt Monthly Mean Demand ===
Month  Train mean (2013-22)  Test mean (2024-26)  Change %
  Jan                1617.0               1813.0      12.1
  Feb                2060.0               2920.0      41.7
  Mar                2919.0               3682.0      26.1
  Apr                4095.0               4807.0      17.4
  May                4768.0               5064.0       6.2
  Jun                5831.0               5755.0      -1.3
  Jul                6539.0               6075.0      -7.1
  Aug                5111.0               5113.0       0.0
  Sep                4523.0               4470.0      -1.2
  Oct                3973.0               3486.0     -12.3
  Nov                3046.0               2879.0      -5.5
  Dec                1919.0               2096.0       9.2

Flagged months:
  April: train=4095  test=4807  change=+17.4%
  June: train=5831  test=5755  change=-1.3%
  July: train=6539  test=6075  change=-7.1%
  August: train=5111  test=5113  chan

## Summary Reference Sheet
All results compiled below for direct use in thesis.

In [12]:
print('=' * 70)
print('NOTEBOOK 14 — COMPLETE REFERENCE SHEET')
print('=' * 70)

print('\n--- TASK 1: SeasonalNaive-364 ---')
print(f"{'Station':<10} {'Val 7d RMSE':>14} {'Test 7d RMSE':>14}")
for s in ['Hirsch', 'Olympia', 'Erhardt']:
    r = sn364_results[s]
    vr = r.get('val_rmse', 614.0)
    print(f"{s:<10} {vr:>14.1f} {r['test_rmse']:>14.1f}")

print('\n--- TASK 2: CV-RMSE (%) ---')
print(cv_df.to_string())
print('\n--- TASK 2: Skill Score (%) vs SeasonalNaive-364 ---')
print(sk_df.to_string())

print('\n--- TASK 3: VIF ---')
for s in ['Hirsch', 'Olympia']:
    print(f'  {s}:')
    print(vif_results[s].to_string(index=False))
    print()

print('--- TASK 3: max_temp coefficient (univariate vs multivariate) ---')
print(coef_df.to_string(index=False))

print('\n--- TASK 4: Diebold-Mariano Tests ---')
print(dm_df[['Pair','Station','DM_stat','p_value','Interpretation']].to_string(index=False))

print('\n--- TASK 5a: Sunshine-only SARIMAX Hirsch ---')
print(f'  Test 7d RMSE = {sun_rmse:.1f}  (report said ~479)')

print('\n--- TASK 5b: Hybrid Corrector Top-5 Importances ---')
for station in STATIONS:
    for hybrid_label, corr_key in [('SARIMAX+XGB', '_corr_sar'), ('Prophet+XGB', '_corr_proph')]:
        subset = imp_df[(imp_df['Station']==station) & (imp_df['Model'].str.contains(hybrid_label.split('+')[0]))]
        feats = ', '.join([f"{r['Feature']}({r['Importance']:.3f})" for _, r in subset.iterrows()])
        print(f'  {station} {hybrid_label}: {feats}')

print('\n--- TASK 5c: Erhardt Monthly Demand ---')
print(monthly_df.to_string(index=False))

print('\nAll CSVs saved to results/tables/nb14_*.csv')

NOTEBOOK 14 — COMPLETE REFERENCE SHEET

--- TASK 1: SeasonalNaive-364 ---
Station       Val 7d RMSE   Test 7d RMSE
Hirsch              614.0          662.1
Olympia             865.9          972.5
Erhardt            1939.7         1906.5

--- TASK 2: CV-RMSE (%) ---
                   Hirsch  Olympia  Erhardt
Model                                      
SARIMAX+wx           31.6     34.5     21.4
Prophet+wx           21.2     24.6     22.0
SARIMAX+XGB          28.1     33.0     18.2
Prophet+XGB          18.8     22.7     19.6
SeasonalNaive-364    44.4     44.2     48.3

--- TASK 2: Skill Score (%) vs SeasonalNaive-364 ---
                   Hirsch  Olympia  Erhardt
Model                                      
SARIMAX+wx           28.9     21.9     55.8
Prophet+wx           52.2     44.2     54.3
SARIMAX+XGB          36.7     25.2     62.2
Prophet+XGB          57.7     48.7     59.4
SeasonalNaive-364     0.0      0.0      0.0

--- TASK 3: VIF ---
  Hirsch:
       Feature  VIF
         sin

---
## Thesis figure: Erhardt monthly mean demand — train vs test (nb14_fig_01)

In [ ]:
# ── nb14_fig_01 — Erhardt monthly demand: training vs test ───────────────────
monthly = pd.read_csv(TABLES / 'nb14_erhardt_monthly.csv')
train_v = monthly['Train mean (2013-22)'].values
test_v  = monthly['Test mean (2024-26)'].values

x = np.arange(12); w = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, train_v, w, label='Train 2013–2022', color=SPLIT_COLORS['Train'], alpha=0.85)
b2 = ax.bar(x + w/2, test_v,  w, label='Test 2024–2026',  color=SPLIT_COLORS['Test'],  alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(MONTHS_SHORT, fontsize=10)
ax.set_ylabel('Mean daily cyclist count'); ax.set_title('Erhardt — Monthly Mean Daily Demand: Train vs Test')
ax.legend()

# Annotate months with notable changes
pct_change = monthly['Change %'].values
for j in range(12):
    pch = pct_change[j]
    if abs(pch) >= 10:
        sign = '↑' if pch > 0 else '↓'
        ax.text(j + w/2, test_v[j] + 30, f'{sign}{abs(pch):.0f}%',
                ha='center', fontsize=7.5,
                color=SPLIT_COLORS['Test'] if pch < 0 else SPLIT_COLORS['Train'])
fig.tight_layout()
savefig('fig_01_erhardt_monthly_demand')


---
## Thesis figure: Erhardt residuals — SARIMAX vs Prophet two-panel (nb14_fig_02)

Requires a quick Prophet refit at Erhardt.

In [ ]:
# ── nb14_fig_02 — Erhardt residuals: SARIMAX (top) vs Prophet+wx+hol (bottom) ─
from pathlib import Path

PREDS_DIR = Path('../results/predictions')
DATA_PATH_LOCAL = Path('../data/processed/master_bike_data.csv')

# SARIMAX residuals from saved predictions
sar = pd.read_csv(PREDS_DIR / 'erhardt_sarimax_preds.csv', parse_dates=['date'])
sar_test = sar[(sar['split'] == 'test') & sar['actual'].notna()].copy()
sar_test['resid'] = sar_test['actual'] - sar_test['pred_full_model']
sar_mean = sar_test['resid'].mean()

# Prophet refit at Erhardt (uses BY_HOLIDAYS already defined earlier in this notebook)
df_local = pd.read_csv(DATA_PATH_LOCAL, parse_dates=['date'])
er = (df_local[df_local['station'] == 'Erhardt']
     [['date', 'total', 'max_temp', 'sunshine_hours', 'precipitation']]
     .dropna(subset=['total'])
     .rename(columns={'date': 'ds', 'total': 'y'}))

trainval = er[(er['ds'] >= '2013-01-01') & (er['ds'] <= '2023-12-31')].copy()
test_er  = er[(er['ds'] >= '2024-01-01') & (er['ds'] <= '2026-04-30')].copy()

m_prop = Prophet(seasonality_mode='multiplicative', holidays=holiday_df,
                 yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
for col in ['max_temp', 'sunshine_hours', 'precipitation']:
    m_prop.add_regressor(col)
m_prop.fit(trainval[['ds', 'y', 'max_temp', 'sunshine_hours', 'precipitation']])

fc = m_prop.predict(test_er[['ds', 'max_temp', 'sunshine_hours', 'precipitation']])
test_er = test_er.set_index('ds')
test_er['yhat'] = fc.set_index('ds')['yhat']
test_er['resid'] = test_er['y'] - test_er['yhat']
prop_mean = test_er['resid'].mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, resid_series, date_series, mean_v, label, color in [
    (axes[0], sar_test['resid'], sar_test['date'], sar_mean,
     'SARIMAX + Fourier K=4 + Weather', SPLIT_COLORS['Train']),
    (axes[1], test_er['resid'], test_er.index, prop_mean,
     'Prophet + Weather + Holidays', SPLIT_COLORS['Val']),
]:
    ax.plot(date_series, resid_series, lw=0.7, color=color, alpha=0.8)
    ax.axhline(0, color='black', lw=1.0, ls='--')
    ax.axvspan(GAP_START, GAP_END, color='#aaa', alpha=0.3, label='Sensor gap')
    ax.axhline(mean_v, color=SPLIT_COLORS['Test'], lw=1.3, ls=':',
               label=f'Mean = {mean_v:.1f}')
    ax.set_ylabel('Residual (actual − pred)', fontsize=9)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(-4200, 4200)
axes[-1].set_xlabel('Date')
fig.suptitle('Erhardt — Daily Forecast Residuals (Test Period 2024–2026)', fontsize=12)
fig.tight_layout()
savefig('fig_02_erhardt_residuals_comparison')


---
## Thesis figure: Erhardt monthly residuals by model (nb14_fig_03)

In [ ]:
# ── nb14_fig_03 — Erhardt monthly mean residuals by model ────────────────────
sar_test2 = sar_test.copy()
sar_test2['month'] = sar_test2['date'].dt.month
sar_by_m  = sar_test2.groupby('month')['resid'].mean()

test_er2  = test_er.dropna(subset=['resid']).copy()
test_er2['month'] = test_er2.index.month
prop_by_m = test_er2.groupby('month')['resid'].mean()

x = np.arange(1, 13); w = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, [sar_by_m.get(m, 0)  for m in range(1, 13)], w,
       label='SARIMAX+wx', color=SPLIT_COLORS['Train'], alpha=0.85)
ax.bar(x + w/2, [prop_by_m.get(m, 0) for m in range(1, 13)], w,
       label='Prophet+wx+hol', color=SPLIT_COLORS['Val'],   alpha=0.85)
ax.axhline(0, color='black', lw=1.0)
ax.set_xticks(x); ax.set_xticklabels(MONTHS_SHORT, fontsize=9)
ax.set_ylabel('Mean monthly residual (actual − forecast)')
ax.set_title('Erhardt — Monthly Mean Residuals by Model (Test Period)')
ax.legend()
fig.tight_layout()
savefig('fig_03_erhardt_monthly_residuals')


---
## Thesis figure: Hybrid feature importance heatmap (nb14_fig_04)

In [ ]:
# ── nb14_fig_04 — Hybrid feature importance heatmap ─────────────────────────
imp = pd.read_csv(TABLES / 'nb14_hybrid_importances.csv')

# Readable feature labels
FEAT_LABELS = {
    'is_holiday': 'Holiday', 'is_weekend': 'Weekend', 'dow': 'DoW',
    'doy': 'DoY', 'month': 'Month', 'max_temp': 'Max Temp',
    'sunshine_hours': 'Sunshine', 'precipitation': 'Precip',
    'sin_1': 'sin1', 'cos_1': 'cos1', 'sin_2': 'sin2', 'cos_2': 'cos2',
    'sin_3': 'sin3', 'cos_3': 'cos3', 'sin_4': 'sin4', 'cos_4': 'cos4',
}

imp['model_short'] = imp['Model'].apply(lambda x: 'SARIMAX+XGB' if 'SARIMAX' in x else 'Prophet+XGB')
imp['feat_label']  = imp['Feature'].map(lambda f: FEAT_LABELS.get(f, f))

# Build pivot: rows = station+model, cols = feature, values = importance
pivot = (imp.groupby(['Station', 'model_short', 'Feature', 'feat_label'])['Importance']
           .max().reset_index()
           .pivot_table(index=['Station', 'model_short'], columns='feat_label',
                        values='Importance', aggfunc='max', fill_value=0))

# Sort columns by max importance across all rows; keep top 10
col_order = pivot.max(axis=0).sort_values(ascending=False).head(10).index.tolist()
pivot = pivot[col_order]

# Reorder rows: Hirsch SARIMAX, Hirsch Prophet, Olympia ..., Erhardt ...
row_order = [('Hirsch', 'SARIMAX+XGB'), ('Hirsch', 'Prophet+XGB'),
             ('Olympia', 'SARIMAX+XGB'), ('Olympia', 'Prophet+XGB'),
             ('Erhardt', 'SARIMAX+XGB'), ('Erhardt', 'Prophet+XGB')]
pivot = pivot.reindex(row_order)

row_labels = [f'{st}\n{m}' for st, m in row_order]

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=pivot.values.max())
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label='Feature importance (gain)')

ax.set_xticks(range(len(col_order))); ax.set_xticklabels(col_order, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8.5)
ax.set_title('XGBoost Corrector Feature Importance — Hybrid Models × Station')

# Annotate cell values
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        tc = 'white' if v > 0.15 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color=tc)

# White dividers between stations (every 2 rows)
for k in [1.5, 3.5]:
    ax.axhline(k, color='white', lw=2.0)

fig.tight_layout()
savefig('fig_04_hybrid_importance_heatmap')


---
## Thesis figure: Hybrid feature importance grouped bar chart (nb14_fig_05)

In [ ]:
# ── nb14_fig_05 — Hybrid feature importance grouped bar chart ────────────────
imp2 = pd.read_csv(TABLES / 'nb14_hybrid_importances.csv')
imp2['model_short'] = imp2['Model'].apply(lambda x: 'SARIMAX+XGB' if 'SARIMAX' in x else 'Prophet+XGB')
imp2['feat_label']  = imp2['Feature'].map(lambda f: FEAT_LABELS.get(f, f))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
model_colors = {'SARIMAX+XGB': SPLIT_COLORS['Train'], 'Prophet+XGB': SPLIT_COLORS['Val']}

for ax, station in zip(axes, ['Hirsch', 'Olympia', 'Erhardt']):
    sub = imp2[imp2['Station'] == station].copy()
    # Get union of top-7 features across both models
    top_feats = (sub.groupby('feat_label')['Importance'].max()
                    .sort_values(ascending=False).head(7).index.tolist())
    sub = sub[sub['feat_label'].isin(top_feats)]

    x = np.arange(len(top_feats)); w = 0.38
    for i, (model, offset) in enumerate([('SARIMAX+XGB', -w/2), ('Prophet+XGB', w/2)]):
        vals = [sub[(sub['model_short'] == model) & (sub['feat_label'] == f)]['Importance']
                    .values[0] if len(sub[(sub['model_short'] == model) &
                    (sub['feat_label'] == f)]) else 0
                for f in top_feats]
        ax.bar(x + offset, vals, w, label=model, color=model_colors[model], alpha=0.85)

    ax.set_xticks(x); ax.set_xticklabels(top_feats, rotation=35, ha='right', fontsize=8)
    ax.set_title(station); ax.set_ylabel('Importance (gain)' if station == 'Hirsch' else '')
    if station == 'Hirsch':
        ax.legend(fontsize=8)

fig.suptitle('Top XGBoost Corrector Features by Station and Hybrid Model', fontsize=12)
fig.tight_layout()
savefig('fig_05_hybrid_importance_bars')


---
## Thesis figure: Skill score bar chart (nb14_fig_06)

In [ ]:
# ── nb14_fig_06 — Skill score bar chart ─────────────────────────────────────
sk = pd.read_csv(TABLES / 'nb14_skill_scores.csv', index_col='Model')
models_plot = ['SARIMAX+wx', 'SARIMAX+XGB', 'Prophet+wx', 'Prophet+XGB']
sk_plot = sk.loc[models_plot]

ST_COLORS = {'Hirsch': '#2ca02c', 'Olympia': '#17becf', 'Erhardt': '#ff7f0e'}
x = np.arange(len(models_plot)); w = 0.26; offsets = [-w, 0, w]

fig, ax = plt.subplots(figsize=(11, 5))
for j, station in enumerate(['Hirsch', 'Olympia', 'Erhardt']):
    vals = sk_plot[station].values
    bars = ax.bar(x + offsets[j], vals, w, label=station,
                  color=ST_COLORS[station], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=7.5, rotation=90)

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['SARIMAX+weather', 'SARIMAX+XGB', 'Prophet+weather', 'Prophet+XGB'],
                   fontsize=10)
ax.set_ylabel('Skill score (% improvement vs Seasonal Naïve-364)')
ax.set_title('Model Skill Scores vs Seasonal Naïve-364 Baseline')
ax.legend(title='Station', fontsize=9)
ax.set_ylim(0, 75)
fig.tight_layout()
savefig('fig_06_skill_scores')
